# Level 1 detector benchmark

Compares candidate detectors on real footage (`Multi_Road_Merged.MP4`, mounted from a
Kaggle Dataset rather than re-uploaded per-run - see below) on Kaggle's T4 - same hardware
Level 1 will actually run on, so this benchmark result transfers directly rather than
needing re-validation on different hardware later. See `PLAN.md` for the full reasoning and
the single-crop result that got us here (VisDrone-pretrained YOLOv8x vs. generic COCO YOLO).

**No ground truth exists for this footage.** Comparison here is: inference time, raw per-class
detection counts, and saved annotated frames for our own visual review - not precision/recall.
Revisit with hand-labeled frames if models end up too close to call visually.

## 1. GPU check

In [ ]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"Device {i}:", torch.cuda.get_device_name(i),
              round(torch.cuda.get_device_properties(i).total_memory / 1e9, 1), "GB")

## 2. Install dependencies

No `-U`, no explicit `torch`/`torchvision` - avoids the CUDA-kernel-mismatch bug found while
setting up the smoke-test notebook (see `KAGGLE.md`). This notebook only needs detection
libraries, not transformers-from-source, so that specific risk is lower here regardless.

In [ ]:
!pip install -q ultralytics sahi huggingface_hub supervision

## 3. Locate the video - mounted from a Kaggle Dataset, not re-uploaded

**Changed from the original plan**: this was going to fetch from a shared Google Drive
folder via `gdown`, but that hit Google's anonymous-download quota mid-hackathon ("too many
users have viewed or downloaded this file recently" - almost certainly other teams hitting
the same shared link). Uploaded the video once as a private Kaggle Dataset instead
(`prithvirajgotepatil/flytbase-hackathon-footage`, added to this kernel's `dataset_sources`
in `kernel-metadata.json`) - Kaggle mounts it read-only at `/kaggle/input/...` automatically,
no download step needed at all, and no dependency on Drive's shared quota.

In [ ]:
import glob

found = glob.glob("/kaggle/input/flytbase-hackathon-footage/**/*.MP4", recursive=True) + \
        glob.glob("/kaggle/input/flytbase-hackathon-footage/**/*.mp4", recursive=True)
assert found, "No .MP4 found under the mounted dataset - check dataset_sources in kernel-metadata.json"
video_path = found[0]
print("Video:", video_path)

## 4. Extract sample frames

~18 frames spread evenly across the clip - enough to judge each model's behaviour without
burning GPU time running every candidate over all ~9,140 frames just to benchmark.

In [ ]:
import cv2
import os

SAMPLES_DIR = "/kaggle/working/samples"
os.makedirs(SAMPLES_DIR, exist_ok=True)
N_SAMPLES = 18

cap = cv2.VideoCapture(video_path)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
sample_indices = [int(i * total_frames / N_SAMPLES) for i in range(N_SAMPLES)]

sample_paths = []
for idx in sample_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, frame = cap.read()
    if not ok:
        continue
    path = f"{SAMPLES_DIR}/frame_{idx:05d}.jpg"
    cv2.imwrite(path, frame)
    sample_paths.append(path)
cap.release()
print(f"Extracted {len(sample_paths)} sample frames from {total_frames} total.")

## 5. Candidate models

`visdrone-yolov8x` already won a single-crop comparison against generic COCO YOLO (see
`PLAN.md`) - included here to confirm that holds at full-frame/full-video scale, alongside a
smaller VisDrone variant (speed tradeoff) and a size-matched generic model (reference point).

In [ ]:
from huggingface_hub import hf_hub_download

CANDIDATES = {
    "visdrone-yolov8x": hf_hub_download(repo_id="dronefreak/visdrone-yolov8x", filename="best.pt"),
    "visdrone-yolov11s": hf_hub_download(repo_id="dronefreak/visdrone-yolov11s", filename="best.pt"),
    # yolo11x-generic (COCO) dropped after v3's run - decisively worse: misses most
    # pedestrians/motorcycles, hallucinates classes (train/boat/airplane) that don't belong
    # in aerial city footage. See PLAN.md.
}
print(CANDIDATES)

## 6. Run each candidate via SAHI sliced inference, on every sample frame

SAHI handles the tile-and-merge logic (slice the 4K frame into overlapping crops, run the
detector on each, merge boxes back to full-frame coordinates with NMS) rather than us
hand-rolling it - this is exactly the tiling need `PLAN.md` flagged after the wide-shot crop
test (vehicles only a few pixels wide in the full frame), and confirmed again by the local
no-tiling quick-track sample (misclassified cars as trucks - see `PLAN.md`).

In [ ]:
import time
from collections import Counter
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

OUTPUT_DIR = "/kaggle/working/benchmark_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results_summary = {}

for name, weights_path in CANDIDATES.items():
    print(f"\n=== {name} ===")
    detection_model = AutoDetectionModel.from_pretrained(
        model_type="ultralytics",
        model_path=weights_path,
        confidence_threshold=0.15,
        device="cuda:0" if torch.cuda.is_available() else "cpu",
    )

    class_counts = Counter()
    times = []
    for path in sample_paths:
        t0 = time.time()
        result = get_sliced_prediction(
            path, detection_model,
            slice_height=768, slice_width=768,
            overlap_height_ratio=0.2, overlap_width_ratio=0.2,
        )
        times.append(time.time() - t0)
        for pred in result.object_prediction_list:
            class_counts[pred.category.name] += 1

        result.export_visuals(export_dir=OUTPUT_DIR, file_name=f"{name}_{os.path.basename(path)}")

    avg_time = sum(times) / len(times)
    results_summary[name] = {"avg_time_s": avg_time, "class_counts": dict(class_counts)}
    print(f"Avg time/frame: {avg_time:.2f}s")
    print(f"Class counts across {len(sample_paths)} frames:", dict(class_counts))

## 7. Comparison table

In [ ]:
import json
print(json.dumps(results_summary, indent=2))

## Next

Pull `/kaggle/working/benchmark_outputs` (via `kaggle kernels output`) for visual review of
annotated frames per model. Decide the detector based on timing + counts + visual quality,
then move to full-video processing + ByteTrack integration.